In [3]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score , precision_score , recall_score,f1_score,classification_report, confusion_matrix
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import matplotlib.pyplot as plt
import seaborn as sns


C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
mlflow.set_tracking_uri("https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow")

In [5]:
import dagshub
dagshub.init(repo_owner='Aayush10671', repo_name='yt-comment-sentiment-analysis', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

Accessing as Aayush10671

Initialized MLflow to track repo "Aayush10671/yt-comment-sentiment-analysis"

Repository Aayush10671/yt-comment-sentiment-analysis initialized!

🏃 View run masked-grouse-322 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/0/runs/9cf50b8b0b77452db70dd2b63191512d
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/0


In [6]:
df = pd.read_csv("preprocessed_data.csv")
df.shape

(36793, 2)

In [7]:
# Drop rows where clean_comment is missing
df = df.dropna(subset=['clean_comment'])
# Remove rows where the comment is empty after stripping
df = df[df['clean_comment'].str.strip() != '']
# Ensure all values are strings (just in case)
df['clean_comment'] = df['clean_comment'].astype(str)

In [8]:
df = df.dropna(subset=['clean_comment'])
df = df[df['clean_comment'].str.strip() != '']
df['clean_comment'] = df['clean_comment'].astype(str)

In [9]:
mlflow.set_experiment("exp-6 hyperparameter tuning")

2026/07/26 00:42:34 INFO mlflow.tracking.fluent: Experiment with name 'exp-6 hyperparameter tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/442ae94d8aaa48f8a81932e0b9856ba2', creation_time=1785006755190, experiment_id='7', last_update_time=1785006755190, lifecycle_stage='active', name='exp-6 hyperparameter tuning', tags={}, workspace='default'>

In [10]:
df['category'] = df['category'].map({0:0 , 1:1 , -1:2})

df = df.dropna(subset = ['category'])

In [ ]:
df.head(5)

,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,2
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [12]:

from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE

ngram_range = (1,3)
max_feature = 1000
vectorizer = TfidfVectorizer(
        ngram_range=ngram_range,
        max_features=max_feature
    )

X = vectorizer.fit_transform(df["clean_comment"])
y = df["category"].values

smote = SMOTE(random_state = 42)

X_resampled, y_resampled = smote.fit_resample(X, y)


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
        X_resampled,
        y_resampled,
        test_size=0.2,
        random_state=42,
    )

In [17]:
import optuna
import mlflow
import mlflow.sklearn
from lightgbm import LGBMClassifier

# -----------------------------
# Log to MLflow
# -----------------------------
def log_model(model_name, model, X_train, X_test, y_train, y_test, params, trial_number):

    with mlflow.start_run(run_name=f"Trial_{trial_number}"):

        mlflow.set_tag("model", model_name)
        mlflow.set_tag("trial_number", trial_number)

        mlflow.log_param("algorithm", model_name)

        for key, value in params.items():
            mlflow.log_param(key, value)

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        report = classification_report(
            y_test,
            y_pred,
            output_dict=True
        )

        for label, metrics in report.items():
            if isinstance(metrics, dict):
                for metric_name, metric_value in metrics.items():
                    mlflow.log_metric(
                        f"{label}_{metric_name}",
                        metric_value
                    )

        mlflow.sklearn.log_model(
            sk_model=model,
            name=f"{model_name}_model"
        )

    return accuracy


# -----------------------------
# Optuna Objective
# -----------------------------
def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "random_state": 42
    }

    model = LGBMClassifier(**params)

    accuracy = log_model(
        model_name="LightGBM",
        model=model,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        params=params,
        trial_number=trial.number
    )

    return accuracy


# -----------------------------
# Run Optuna
# -----------------------------
def run_optuna_experiment(n_trials=20):

    study = optuna.create_study(direction="maximize")

    study.optimize(objective, n_trials=n_trials)

    print("=" * 50)
    print("Best Accuracy :", study.best_value)
    print("Best Parameters:")
    print(study.best_params)
    print("=" * 50)

    return study


# -----------------------------
# Start Hyperparameter Tuning
# -----------------------------
study = run_optuna_experiment(n_trials=20)

[I 2026-07-26 00:58:41,132] A new study created in memory with name: no-name-efa1d6de-7b03-4a19-b150-3e59e8c829ad


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.431888 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 98898
[LightGBM] [Info] Number of data points in the train set: 37848, number of used features: 974
[LightGBM] [Info] Start training from score -1.093552
[LightGBM] [Info] Start training from score -1.104097
[LightGBM] [Info] Start training from score -1.098216
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] N

C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/07/26 00:59:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Trial_0 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/7/runs/c029cc5a7e554bfea09699cba0a14f62
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/7


[I 2026-07-26 00:59:52,545] Trial 0 finished with value: 0.8124075248361868 and parameters: {'n_estimators': 375, 'max_depth': 8, 'learning_rate': 0.17080056566672677, 'num_leaves': 61, 'subsample': 0.6301420284180647, 'colsample_bytree': 0.7941424862438891}. Best is trial 0 with value: 0.8124075248361868.


...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.085867 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 98898
[LightGBM] [Info] Number of data points in the train set: 37848, number of used features: 974
[LightGBM] [Info] Start training from score -1.093552
[LightGBM] [Info] Start training from score -1.104097
[LightGBM] [Info] Start training from score -1.098216
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] N

C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/07/26 01:32:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Trial_19 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/7/runs/3a49b58a071a4a16bced2716a72c7ed4
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/7


[I 2026-07-26 01:32:47,095] Trial 19 finished with value: 0.8101881209046713 and parameters: {'n_estimators': 268, 'max_depth': 10, 'learning_rate': 0.24115861225547114, 'num_leaves': 68, 'subsample': 0.7179869058830504, 'colsample_bytree': 0.7501651774520554}. Best is trial 4 with value: 0.8137814415556964.


Best Accuracy : 0.8137814415556964
Best Parameters:
{'n_estimators': 222, 'max_depth': 11, 'learning_rate': 0.1896241135620414, 'num_leaves': 78, 'subsample': 0.6750162146487924, 'colsample_bytree': 0.9043761274518365}
